# 第九章：PyTorch 的模型部署

这一章讲 PyTorch 模型如何导出为 ONNX，并用 ONNX Runtime 做推理。这个 notebook 使用一个极小模型完整跑通：定义模型、导出 ONNX、检查模型、ONNX Runtime 推理、和 PyTorch 输出对比。

## 1. 导入库并定义小模型

部署时通常要先把模型切到 `eval()`，避免 Dropout、BatchNorm 等训练行为影响推理。

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

torch.manual_seed(42)

class TinyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):
        return self.net(x)

model = TinyRegressor().eval()
dummy_input = torch.randn(1, 4)

with torch.no_grad():
    torch_output = model(dummy_input)

print("输入：", dummy_input)
print("PyTorch 输出：", torch_output)


## 2. 导出为 ONNX

`dummy_input` 的作用是告诉导出器：模型输入长什么样。这里还设置了动态 batch 维度，方便未来一次推理多条数据。

In [ ]:
tmpdir = tempfile.TemporaryDirectory()
onnx_path = Path(tmpdir.name) / "tiny_regressor.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)

print("ONNX 文件路径：", onnx_path)
print("ONNX 文件大小：", onnx_path.stat().st_size, "bytes")


## 3. 检查 ONNX 模型是否合法

`onnx.checker.check_model` 会检查模型结构是否符合 ONNX 标准。

In [ ]:
onnx_model = onnx.load(onnx_path)
try:
    onnx.checker.check_model(onnx_model)
except onnx.checker.ValidationError as exc:
    print("ONNX 模型无效：", exc)
else:
    print("ONNX 模型检查通过。")


## 4. 用 ONNX Runtime 推理

ONNX Runtime 接收的是 NumPy array，不是 PyTorch tensor，所以要先转换。

In [ ]:
def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_name = ort_session.get_inputs()[0].name
output_name = ort_session.get_outputs()[0].name

ort_inputs = {input_name: to_numpy(dummy_input)}
ort_output = ort_session.run([output_name], ort_inputs)[0]

print("输入名：", input_name)
print("输出名：", output_name)
print("ONNX Runtime 输出：")
print(ort_output)


## 5. 对比 PyTorch 和 ONNX Runtime 输出

导出后最重要的 sanity check：同一个输入，两边输出应当非常接近。

In [ ]:
torch_np = to_numpy(torch_output)
max_abs_diff = np.max(np.abs(torch_np - ort_output))

print("PyTorch 输出：", torch_np)
print("ONNX Runtime 输出：", ort_output)
print("最大绝对差异：", max_abs_diff)
print("是否足够接近：", np.allclose(torch_np, ort_output, atol=1e-6))


## 6. 动态 batch 验证

因为导出时设置了动态 batch，这里换成 3 条输入也能推理。

In [ ]:
batch_input = torch.randn(3, 4)
with torch.no_grad():
    torch_batch_output = model(batch_input)

ort_batch_output = ort_session.run([output_name], {input_name: to_numpy(batch_input)})[0]
print("batch 输入形状：", batch_input.shape)
print("PyTorch batch 输出形状：", torch_batch_output.shape)
print("ONNX Runtime batch 输出形状：", ort_batch_output.shape)
print("batch 最大绝对差异：", np.max(np.abs(to_numpy(torch_batch_output) - ort_batch_output)))


## 7. 清理临时文件

这个 notebook 用临时目录演示，不会在仓库里留下 `.onnx` 文件。真实项目可以把 `onnx_path` 改成固定文件名，比如 `model.onnx`。

In [ ]:
tmpdir.cleanup()
print("临时 ONNX 文件已清理。")


## 8. 小结

部署流水线可以记成：

1. PyTorch 模型 `model.eval()`。
2. 准备 `dummy_input`。
3. `torch.onnx.export(...)` 导出 `.onnx`。
4. `onnx.checker.check_model(...)` 检查。
5. `onnxruntime.InferenceSession(...)` 推理。
6. 对比 PyTorch 和 ONNX Runtime 输出是否接近。